In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import matplotlib
from numpy import random
import copy
import scipy
import os
import sys

In [ ]:
matplotlib.rc("font", **{"family":"serif","serif":["Computer Modern Roman"]})
matplotlib.rc("text", usetex=True)

# load data

In [ ]:
# simulation parameters
# starfish model

numx = 30
numy = 30
N = numx*numy
R = 0.5
nrec = 1000
n_j = nrec
wnum = int(n_j/2)+1

spacing = 1.2
dx = 0.4*spacing
dy = 0.4*spacing
Lx = spacing * 2 * R * numx
Ly = 0.5 * np.sqrt(3) * numy * spacing * 2 * R
xnum = round(Lx/dx)
ynum = round(Ly/dy)
dkx = 2*np.pi/Lx
dky = 2*np.pi/Ly
dt = 0.1
dw = 2*np.pi/(dt*n_j)

# frequency domain 
kxrange = np.zeros(xnum)
kyrange = np.zeros(ynum)
wrange = dw*np.arange(wnum)

for i in range(xnum):
    kxrange[i] = -np.pi/dx + dkx*i
for j in range(ynum):
    kyrange[j] = -np.pi/dy + dky*j
    
kx,ky = np.meshgrid(kxrange,kyrange,indexing='ij')


In [ ]:
# when dx was 0.5*spacing
#Gpoint = np.array([30,26],dtype=int)
#Mpoint = np.array([45,33],dtype=int)
#Kpoint = np.array([40,41],dtype=int)

# when dx was 0.4 spacing
Gpoint = np.array([37,32],dtype=int)
Mpoint = np.array([53,40],dtype=int)
Kpoint = np.array([48,47],dtype=int)

# when dx was 0.3 spacing
#Gpoint = np.array([50,43],dtype=int)
#Kpoint = np.array([60,58],dtype=int)
#Mpoint = np.array([65,51],dtype=int)

In [ ]:
base_dir = 'directory of data for alpha=0.79 and vsig=1'

file_name = 'Cll_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Cll_1 = np.fromfile(f,np.float64)
Cll_1 = (1/N)*Cll_1.reshape((xnum,ynum,wnum))

file_name = 'Ctt_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Ctt_1 = np.fromfile(f,np.float64)
Ctt_1 = (1/N)*Ctt_1.reshape((xnum,ynum,wnum))


base_dir = 'directory of data for alpha=2.63 and vsig=0.02'

file_name = 'Cll_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Cll_2 = np.fromfile(f,np.float64)
Cll_2 = (1/N)*Cll_2.reshape((xnum,ynum,wnum))

file_name = 'Ctt_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Ctt_2 = np.fromfile(f,np.float64)
Ctt_2 = (1/N)*Ctt_2.reshape((xnum,ynum,wnum))


In [ ]:
ikx = Mpoint[0]
iky = Mpoint[1]

C1full = copy.copy(Cll_1) + copy.copy(Ctt_1)
C1full_q = copy.copy(C1full[ikx,iky,:])

C2full = copy.copy(Cll_2) + copy.copy(Ctt_2)
C2full_q = copy.copy(C2full[ikx,iky,:])


# coefficient of variation

In [ ]:
mu = 0 # mean
var = 0 # variance
Csum = 0
nonzeroCnum = 0

for i in range(wnum):
    if C1full_q[i]>0:
        Csum += C1full_q[i]
        nonzeroCnum += 1

for i in range(wnum):
    mu += wrange[i]*C1full_q[i]/Csum
    
for i in range(wnum):
    var += C1full_q[i]*((wrange[i]-mu)**2)/((nonzeroCnum-1)*Csum/nonzeroCnum)
    
CV1 = np.sqrt(var)/mu


In [ ]:
mu = 0 # mean
var = 0 # variance
Csum = 0
nonzeroCnum = 0

for i in range(wnum):
    if C2full_q[i]>0:
        Csum += C2full_q[i]
        nonzeroCnum += 1

for i in range(wnum):
    mu += wrange[i]*C2full_q[i]/Csum
    
for i in range(wnum):
    var += C2full_q[i]*((wrange[i]-mu)**2)/((nonzeroCnum-1)*Csum/nonzeroCnum)
    
CV2 = np.sqrt(var)/mu


# plot current correlation function at M point with CV

In [ ]:
%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(15,5))
ax1 = fig.add_subplot(1,2,1)
ax2 = fig.add_subplot(1,2,2)

ax1.plot(wrange,C1full_q,'bo')
ax1.text(0.7*wrange[-1],0.9*np.max(C1full_q),'CV = %s' % ("%.4f"%(CV1)),fontsize=20)
ax1.set_xlabel('$\omega$', fontsize=25)
ax1.set_ylabel('$C_{M}$', fontsize=25)
ax1.text(-6.5,220,'(a)',fontsize=25)
ax1.tick_params(labelsize=13)

ax2.plot(wrange,C2full_q,'bo')
ax2.text(0.7*wrange[-1],0.9*np.max(C2full_q),'CV = %s' % ("%.4f"%(CV2)),fontsize=20)
ax2.set_xlabel('$\omega$', fontsize=25)
ax2.set_ylabel('$C_{M}$', fontsize=25)
ax2.text(-6.5,0.147,'(b)',fontsize=25)
ax2.tick_params(labelsize=13)

plt.show()
#fig.savefig('CM_example_v1.pdf',dpi=fig.dpi,bbox_inches='tight')


# double axis plots of max C at M point and CV

In [ ]:
# the values of max C at M point and CV for different paramters have been obtained
# from performing the above analysis of current correlation functions and CVs for those parameters

vsig_f1 = np.array([0.02,0.04,0.06,0.08,0.1])
CM_f1 = np.array([0.13,0.5,1.1,2.1,3.2])
CV_f1 = np.array([0.5232,0.5235,0.5274,0.5252,0.5250])

f0_vsig1 = np.array([0.3,0.35,0.4,0.45,0.5])
kL = 1.9
alpha_vsig1 = f0_vsig1*5/kL
CM_vsig1 = np.array([110,113,120,123,130])
CV_vsig1 = np.array([0.7295,0.7166,0.6999,0.6870,0.6668])

%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(15,5))
ax1 = fig.add_subplot(1,2,1)
ax2 = fig.add_subplot(1,2,2)
ax3 = ax1.twinx()
ax4 = ax2.twinx()

# first figure
color = '#0072b2'
ax1.plot(alpha_vsig1,CM_vsig1,'o-',color=color,markersize=10)
ax1.plot([alpha_vsig1[0],alpha_vsig1[-1]],[0.5,0.5],'--',color=color)
ax1.set_xlabel(r'$\alpha$',fontsize=25)
ax1.set_ylabel('$C_{M}$',color=color,fontsize=25)
ax1.tick_params(axis='y',labelcolor=color,labelsize=15)
ax1.tick_params(axis='x',labelsize=15)
ax1.text(0.683,125,'(a)',fontsize=25)

color = '#d55e00'
ax3.plot(alpha_vsig1,CV_vsig1,'^-',color=color,markersize=10)
ax3.plot([alpha_vsig1[0],alpha_vsig1[-1]],[0.7,0.7],linestyle='dotted',color=color)
ax3.set_ylabel('$CV$',color=color,fontsize=25)
ax3.tick_params(axis='y',labelcolor=color,labelsize=15)

# second figure
color = '#0072b2'
ax2.plot(vsig_f1,CM_f1,'o-',color=color,markersize=10)
ax2.plot([vsig_f1[0],vsig_f1[-1]],[0.5,0.5],'--',color=color)
ax2.set_xlabel('$v_{\sigma}$',fontsize=25)
ax2.set_ylabel('$C_{M}$',color=color,fontsize=25)
ax2.tick_params(axis='y',labelcolor=color,labelsize=15)
ax2.tick_params(axis='x',labelsize=15)
ax2.text(0.0035,3.1,'(b)',fontsize=25)

#color = '#e69f00'
color = '#d55e00'
ax4.plot(vsig_f1,CV_f1,'^-',color=color,markersize=10)
ax4.plot([vsig_f1[0],vsig_f1[-1]],[0.7,0.7],linestyle='dotted',color=color)
ax4.set_ylabel('$CV$',color=color,fontsize=25)
ax4.tick_params(axis='y',labelcolor=color,labelsize=15)

fig.tight_layout()
plt.show()
#fig.savefig('CM_wave_v2.pdf',dpi=fig.dpi,bbox_inches='tight')


# compare decayed wave regimes

In [ ]:
# f0 = 1.0 but signal too weak (vsig = 0.02)

base_dir = 'directory for data with alpha=2.63, f0=1 and vsig=0.02'

file_name = 'Cll_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Cll_f1vsig002 = np.fromfile(f,np.float64)
Cll_f1vsig002 = (1/N)*Cll_f1vsig002.reshape((xnum,ynum,wnum))

file_name = 'Ctt_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Ctt_f1vsig002 = np.fromfile(f,np.float64)
Ctt_f1vsig002 = (1/N)*Ctt_f1vsig002.reshape((xnum,ynum,wnum))


# f0 = 0.3 vsig = 3.5 near melting

base_dir = 'directory for data with alpha=0.79, f0=0.3 and vsig=3.5'

file_name = 'Cll_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Cll_f03vsig3p5 = np.fromfile(f,np.float64)
Cll_f03vsig3p5 = (1/N)*Cll_f03vsig3p5.reshape((xnum,ynum,wnum))

file_name = 'Ctt_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Ctt_f03vsig3p5 = np.fromfile(f,np.float64)
Ctt_f03vsig3p5 = (1/N)*Ctt_f03vsig3p5.reshape((xnum,ynum,wnum))


# f0 = 0.3 vsig = 0.1 far from melting

base_dir = 'directory for data with alpha=0.79, f0=0.3 and vsig=0.1'

file_name = 'Cll_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Cll_f03vsig01 = np.fromfile(f,np.float64)
Cll_f03vsig01 = (1/N)*Cll_f03vsig01.reshape((xnum,ynum,wnum))

file_name = 'Ctt_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Ctt_f03vsig01 = np.fromfile(f,np.float64)
Ctt_f03vsig01 = (1/N)*Ctt_f03vsig01.reshape((xnum,ynum,wnum))


In [ ]:
# 2d matrix whose elements are the max value of C at particular (qx,qy)
# preserve the sign of Cltr and Clti

#Cll = copy.copy(Cll_f1vsig002)
#Ctt = copy.copy(Ctt_f1vsig002)
#Cll = copy.copy(Cll_f03vsig3p5)
#Ctt = copy.copy(Ctt_f03vsig3p5)
Cll = copy.copy(Cll_f03vsig01)
Ctt = copy.copy(Ctt_f03vsig01)

Cfull_copy = copy.copy(Cll) + copy.copy(Ctt)

# get rid of signal from self-circling
Cfull_copy[:,:,10:20]=0
Cfull_copy[:,:,25:35]=0
Cfull_copy[:,:,40:50]=0

Cfull_peak = np.zeros([xnum,ynum])
wfull_peak = np.zeros([xnum,ynum])

fromhere = 1

for i in range(xnum):
    for j in range(ynum):
        Cfull_thisq = Cfull_copy[i,j,fromhere:]
        
        Cfull_peak[i,j] = np.max(Cfull_thisq)
        idx_wfullmax = np.where(Cfull_thisq==np.max(Cfull_thisq))
        idx_wfullmax = int(idx_wfullmax[0][0]) + fromhere
        wfull_peak[i,j] = wrange[idx_wfullmax]
        
#Cfull_f1vsig002 = copy.copy(Cfull_peak)
#wfull_f1vsig002 = copy.copy(wfull_peak)
#Cfull_f03vsig3p5 = copy.copy(Cfull_peak)
#wfull_f03vsig3p5 = copy.copy(wfull_peak)
Cfull_f03vsig01 = copy.copy(Cfull_peak)
wfull_f03vsig01 = copy.copy(wfull_peak)
        

In [ ]:
#Cll = copy.copy(Cll_f1vsig002)
#Ctt = copy.copy(Ctt_f1vsig002)
#Cll = copy.copy(Cll_f03vsig3p5)
#Ctt = copy.copy(Ctt_f03vsig3p5)
Cll = copy.copy(Cll_f03vsig01)
Ctt = copy.copy(Ctt_f03vsig01)

# along the path in the first Brillouin zone Gamma -> K -> M -> Gamma
Cll_GtoK = copy.copy(Cll)
Cll_GtoK = Cll_GtoK[GtoK[0,:],GtoK[1,:],:]
Cll_KtoM = copy.copy(Cll)
Cll_KtoM = Cll_KtoM[KtoM[0,:],KtoM[1,:],:]
Cll_MtoG = copy.copy(Cll)
Cll_MtoG = Cll_MtoG[MtoG[0,:],MtoG[1,:],:]
Cll_GKMG = np.concatenate((Cll_GtoK,Cll_KtoM,Cll_MtoG),axis=0)

Ctt_GtoK = copy.copy(Ctt)
Ctt_GtoK = Ctt_GtoK[GtoK[0,:],GtoK[1,:],:]
Ctt_KtoM = copy.copy(Ctt)
Ctt_KtoM = Ctt_KtoM[KtoM[0,:],KtoM[1,:],:]
Ctt_MtoG = copy.copy(Ctt)
Ctt_MtoG = Ctt_MtoG[MtoG[0,:],MtoG[1,:],:]
Ctt_GKMG = np.concatenate((Ctt_GtoK,Ctt_KtoM,Ctt_MtoG),axis=0)

qnum = np.size(Cll_GKMG,axis=0)
q_band,w_band = np.meshgrid(np.arange(qnum),wrange,indexing='ij')

fromthisw = 1 # not include w=0
uptothisw = -1 # high frequency w is not relevant
w_band = w_band[:,fromthisw:uptothisw] # only take w>0.
q_band = q_band[:,fromthisw:uptothisw]
Cll_band = Cll_GKMG[:,fromthisw:uptothisw]
Ctt_band = Ctt_GKMG[:,fromthisw:uptothisw]

# get rid of signal from self-circling
Cll_noSC = copy.copy(Cll_band)
Cll_noSC[:,10:20]=0
Cll_noSC[:,25:35]=0
Cll_noSC[:,40:50]=0

Ctt_noSC = copy.copy(Ctt_band)
Ctt_noSC[:,10:20]=0
Ctt_noSC[:,25:35]=0
Ctt_noSC[:,40:50]=0

C_band = copy.copy(Cll_noSC)+copy.copy(Ctt_noSC)

#Cband_f1vsig002 = copy.copy(C_band)
#Cband_f03vsig3p5 = copy.copy(C_band)
Cband_f03vsig01 = copy.copy(C_band)


# plot all together

In [ ]:
tickloc = [0,np.size(GtoK,axis=1),np.size(GtoK,axis=1)+np.size(KtoM,axis=1)-1,np.size(GtoK,axis=1)+np.size(KtoM,axis=1)+np.size(MtoG,axis=1)-1]

%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(36,18))
ax1 = fig.add_subplot(2,3,1,projection='3d')
ax2 = fig.add_subplot(2,3,2,projection='3d')
ax3 = fig.add_subplot(2,3,3,projection='3d')
ax4 = fig.add_subplot(2,3,4)
ax5 = fig.add_subplot(2,3,5)
ax6 = fig.add_subplot(2,3,6)
#fig.tight_layout()

# first plot
img1 = ax1.scatter3D(kx,ky,wfull_f03vsig3p5,c=Cfull_f03vsig3p5,cmap = 'viridis',rasterized=True)

ax1.plot([p1x,p2x,p3x,p4x,p5x,p6x,p1x],[p1y,p2y,p3y,p4y,p5y,p6y,p1y],zs=0,zdir='z',color='k')
ax1.plot([bz1x,bz2x,bz3x,bz4x,bz5x,bz6x,bz1x],[bz1y,bz2y,bz3y,bz4y,bz5y,bz6y,bz1y],zs=0,zdir='z',color='k')
ax1.plot([kx[Gpoint[0],Gpoint[1]],kx[Kpoint[0],Kpoint[1]],kx[Mpoint[0],Mpoint[1]],kx[Gpoint[0],Gpoint[1]]],[ky[Gpoint[0],Gpoint[1]],ky[Kpoint[0],Kpoint[1]],ky[Mpoint[0],Mpoint[1]],ky[Gpoint[0],Gpoint[1]]],zs=0,zdir='z',color='r',linestyle='--')

cb1 = ax1.figure.colorbar(img1,shrink=0.7,pad=0.1)
ax1.set_xlabel('$q_x$', fontsize=40, labelpad=10)
ax1.set_ylabel('$q_y$', fontsize=40, labelpad=10)
ax1.set_zlabel('$\omega$', fontsize=40, labelpad=10)
ax1.text2D(-0.108,0.07,'(a)',fontsize=40,va='top',ha='right')
cb1.set_label('C',fontsize=40)
ax1.tick_params(labelsize=20)
cb1.ax.tick_params(labelsize=20)

# second plot
img2 = ax2.scatter3D(kx,ky,wfull_f03vsig01,c=Cfull_f03vsig01,cmap = 'viridis',rasterized=True)

ax2.plot([p1x,p2x,p3x,p4x,p5x,p6x,p1x],[p1y,p2y,p3y,p4y,p5y,p6y,p1y],zs=0,zdir='z',color='k')
ax2.plot([bz1x,bz2x,bz3x,bz4x,bz5x,bz6x,bz1x],[bz1y,bz2y,bz3y,bz4y,bz5y,bz6y,bz1y],zs=0,zdir='z',color='k')
ax2.plot([kx[Gpoint[0],Gpoint[1]],kx[Kpoint[0],Kpoint[1]],kx[Mpoint[0],Mpoint[1]],kx[Gpoint[0],Gpoint[1]]],[ky[Gpoint[0],Gpoint[1]],ky[Kpoint[0],Kpoint[1]],ky[Mpoint[0],Mpoint[1]],ky[Gpoint[0],Gpoint[1]]],zs=0,zdir='z',color='r',linestyle='--')

cb2 = ax2.figure.colorbar(img2,shrink=0.7,pad=0.1)
ax2.set_xlabel('$q_x$', fontsize=40, labelpad=10)
ax2.set_ylabel('$q_y$', fontsize=40, labelpad=10)
ax2.set_zlabel('$\omega$', fontsize=40, labelpad=10)
ax2.text2D(-0.108,0.07,'(b)',fontsize=40,va='top',ha='right')
cb2.set_label('C',fontsize=40)
ax2.tick_params(labelsize=20)
cb2.ax.tick_params(labelsize=20)

# third plot
img3 = ax3.scatter3D(kx,ky,wfull_f1vsig002,c=Cfull_f1vsig002,cmap = 'viridis',rasterized=True)

ax3.plot([p1x,p2x,p3x,p4x,p5x,p6x,p1x],[p1y,p2y,p3y,p4y,p5y,p6y,p1y],zs=0,zdir='z',color='k')
ax3.plot([bz1x,bz2x,bz3x,bz4x,bz5x,bz6x,bz1x],[bz1y,bz2y,bz3y,bz4y,bz5y,bz6y,bz1y],zs=0,zdir='z',color='k')
ax3.plot([kx[Gpoint[0],Gpoint[1]],kx[Kpoint[0],Kpoint[1]],kx[Mpoint[0],Mpoint[1]],kx[Gpoint[0],Gpoint[1]]],[ky[Gpoint[0],Gpoint[1]],ky[Kpoint[0],Kpoint[1]],ky[Mpoint[0],Mpoint[1]],ky[Gpoint[0],Gpoint[1]]],zs=0,zdir='z',color='r',linestyle='--')

cb3 = ax3.figure.colorbar(img3,shrink=0.7,pad=0.1)
ax3.set_xlabel('$q_x$', fontsize=40, labelpad=10)
ax3.set_ylabel('$q_y$', fontsize=40, labelpad=10)
ax3.set_zlabel('$\omega$', fontsize=40, labelpad=10)
ax3.text2D(-0.105,0.07,'(c)',fontsize=40,va='top',ha='right')
cb3.set_label('C',fontsize=40)
ax3.tick_params(labelsize=20)
cb3.ax.tick_params(labelsize=20)

# fourth plot
img4 = ax4.contourf(q_band,w_band,Cband_f03vsig3p5,levels=100)

cb4 = fig.colorbar(img4,ax=ax4)
ax4.set_ylabel('$\omega$',fontsize=40)
cb4.set_label('C',fontsize=40)
ax4.set_ylim([0,31])
ax4.text(-2.5,31.0,'(d)',fontsize=40,va='top',ha='right')
ax4.tick_params(labelsize=20)
cb4.ax.tick_params(labelsize=20)
ax4.set_xticks(tickloc,["$\Gamma$","K","M","$\Gamma$"],fontsize=40)

# fifth plot
img5 = ax5.contourf(q_band,w_band,Cband_f03vsig01,levels=100)

cb5 = fig.colorbar(img5,ax=ax5)
ax5.set_ylabel('$\omega$',fontsize=40)
cb5.set_label('C',fontsize=40)
ax5.set_ylim([0,31])
ax5.text(-2.5,31.0,'(e)',fontsize=40,va='top',ha='right')
ax5.tick_params(labelsize=20)
cb5.ax.tick_params(labelsize=20)
ax5.set_xticks(tickloc,["$\Gamma$","K","M","$\Gamma$"],fontsize=40)

# sixth plot
img6 = ax6.contourf(q_band,w_band,Cband_f1vsig002,levels=100)

cb6 = fig.colorbar(img6,ax=ax6)
ax6.set_ylabel('$\omega$',fontsize=40)
cb6.set_label('C',fontsize=40)
ax6.set_ylim([0,31])
ax6.text(-2.5,31.0,'(f)',fontsize=40,va='top',ha='right')
ax6.tick_params(labelsize=20)
cb6.ax.tick_params(labelsize=20)
ax6.set_xticks(tickloc,["$\Gamma$","K","M","$\Gamma$"],fontsize=40)

# to avoid the weird white lines appearing in pdf file of the plot
for c in img4.collections:
    c.set_edgecolor("face")

for c in img5.collections:
    c.set_edgecolor("face")
    
for c in img6.collections:
    c.set_edgecolor("face")

#plt.savefig('nowave_comp.pdf',dpi=200,bbox_inches='tight')

fig.tight_layout()
plt.show()
